# Speculative Decoding -- Draft, Verify, Repeat

Autoregressive decoding is serial. Each token waits for previous one. Speculative decoding breaks the chain: a cheeap model drafts N tokens, the expensive model verifies all N in one forward pass, When the draft is right you paid one big forward for N generations.

## Problem Definition

Trade a small amount of extra GPU memory (draft model) for 2-4x lower decode latency.
The trick has to perserve the distribution, concurrently, guarantees that the output sequence is identically distributed to what the big model would have produced on its own.

Four families of draft-verifier pairs:

1. Vanilla speculative.   Seprate draft model + verifier.
2. Medusa.  Multiple decoding heads on the verifier perdict positions t+1 ... t+k in parallel. No separate draft model.
3. EAGLE.   Lightweight draft that reuses the verifier's hidden states.
4. Lookahead decoding.  Jacobi iteration; no draft model required at all. Self-speculation, Niche but dependency-free.

Every production inference stack in 2026 ships speculative decoding by default.

## Basic Concept

Given a verifier M_q and a cheaper draft M_p

1. Let x1...xk be the prefix already deocded.
2. **Draft:** Use M_p to autoregressively propose d_{k+1}...d_{k+N} with draft probabilities p_{1}...p_{N}.
3. **Verifiy in parallel:** Run M_q once on x_1...x_k, d_{k+1}...d_{k+N}, getting verifier probabilities q_{1}...q{N+1}
4. **Accept/reject each draft token left to right** For each i, accept with probability min(1, q_i(d_i) / p_i(d_i))
5. On first rejection at postion j, sample t_j from the residual distribution (q_j - p_j)_ + normalized, all drafts after j are discarded.
6. On accept all N, sample one extra token t_{N+1} from q_{N+1}

### Depencies

* How well the draft approximates the verifier. Same family/ Same training data.
* Decodeing Strategy. Greedy draft against greedy verifier. Temperature sampling: harder to match.
* Task type. Code and structured output accept more (predictable); free-from creative writing accepts less.

## Medusa -- drafts without a draft model

Medusa replaces the draft model with extra output heads on the verifier. At position t:
```
Shared trunk --> hidden h_t
  |--- head_0: predict token at t+1
  |--- head_1: predict token at t+2
  |--- head_2: predict token at t+3
  ...
```

